# Trenowanie modeli 

## Wczytanie zbioru danych

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Dodanie katalogu głównego do ścieżek
root_dir = Path("..").resolve()
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.config import DATASET_2022_PATH

# Wczytanie gotowego zbioru
df = pd.read_parquet(DATASET_2022_PATH)

print(f"Załadowano obserwacji: {len(df):,}")
print(f"Liczba stacji: {df['station'].nunique()}")
df.head()

Załadowano obserwacji: 123,694
Liczba stacji: 15


,timestamp,station,data_source,lat,lon,altitude,urban_fraction_500m,sin_hour,cos_hour,sin_doy,cos_doy,temp_era5,temp_ground,guminski_season,delta_temp,hour
0,2022-01-01,Aleksandrowice,lcs,50.081104,19.764215,242.385681,0.2129,0.0,1.0,0.017202,0.999852,8.454773,8.5,zima,0.045227,0
1,2022-01-01,Biezanow Prokocim,lcs,50.019761,20.052830,208.093094,0.5512,0.0,1.0,0.017202,0.999852,8.989929,9.1,zima,0.110071,0
2,2022-01-01,Debniki,lcs,50.033099,19.895774,214.787842,0.3366,0.0,1.0,0.017202,0.999852,8.778992,8.9,zima,0.121008,0
3,2022-01-01,Grabie,lcs,50.039521,20.134673,197.447372,0.1164,0.0,1.0,0.017202,0.999852,8.989929,9.2,zima,0.210071,0
4,2022-01-01,Krakow-Balice,imgw,50.077700,19.784800,238.000000,0.2566,0.0,1.0,0.017202,0.999852,8.454773,9.2,zima,0.745227,0


## Podział danych na zbiór uczący i testowy

In [ ]:
from src.config import TRAIN_DATA_PATH, TEST_DATA_PATH

df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date

# Kolejność pór roku Gumińskiego
SEASONS_ORDER = ['zima', 'przedwiosnie', 'wiosna', 'lato', 'jesien', 'przedzimie']

test_days = []
for season in SEASONS_ORDER:
    season_df = df[df['guminski_season'] == season]
    
    if season == 'zima':
        zima_early = season_df[season_df['timestamp'].dt.month <= 3]
        days = sorted(zima_early['date'].unique())
        last_7 = days[-7:]
        print(f"- {'Zima (luty)':<18}: od {last_7[0]} do {last_7[-1]} (7 dni)")
    else:
        days = sorted(season_df['date'].unique())
        last_7 = days[-7:]
        print(f"- {season.capitalize():<18}: od {last_7[0]} do {last_7[-1]} (7 dni)")
        
    test_days.extend(last_7)

# Flaga podziału na zbiory
is_test = df['date'].isin(test_days)
df_train = df[~is_test].drop(columns=['date']).copy().reset_index(drop=True)
df_test = df[is_test].drop(columns=['date']).copy().reset_index(drop=True)

# Zapis do plików Parquet
df_train.to_parquet(TRAIN_DATA_PATH, index=False)
df_test.to_parquet(TEST_DATA_PATH, index=False)

print("\nBILANS PODZIAŁU DANYCH")
print(f"Liczba wierszy łącznie: {len(df):,}")
print(f"Zbiór TRENINGOWY:       {len(df_train):,} ({len(df_train) / len(df) * 100:.1f}%)")
print(f"Zbiór TESTOWY (42 dni): {len(df_test):,} ({len(df_test) / len(df) * 100:.1f}%)")

- Zima (luty)       : od 2022-02-05 do 2022-02-11 (7 dni)
- Przedwiosnie      : od 2022-03-18 do 2022-03-24 (7 dni)
- Wiosna            : od 2022-05-15 do 2022-05-21 (7 dni)
- Lato              : od 2022-09-11 do 2022-09-17 (7 dni)
- Jesien            : od 2022-11-09 do 2022-11-15 (7 dni)
- Przedzimie        : od 2022-12-20 do 2022-12-26 (7 dni)

BILANS PODZIAŁU DANYCH
Liczba wierszy łącznie: 123,694
Zbiór TRENINGOWY:       109,456 (88.5%)
Zbiór TESTOWY (42 dni): 14,238 (11.5%)


## Baseline models

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn,linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

features = [
    'temp_era5',
    'altitude',
    'urban_fraction_500m',
    'sin_hour',
    'cos_hour',
    'sin_doy',
    'cos_doy'
]


X_train = df_train[features].values
y_train = (df_train['temp_ground'] - df_train['temp_era5']).values

X_test = df_test[features].values
y_test = (df_test['temp_ground'] - df_test['temp_era5']).values

